In [15]:
# import libraries
import numpy as np
import pandas as pd
import os
import time
import yaml
from pathlib import Path
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from multiprocessing import Pool, cpu_count

In [16]:
def load_config(config_path='../configs/preprocessing.yml'):
    # Guard check - Config Exists
    config_path = Path(config_path)
    if not config_path.exists():
        raise FileNotFoundError(f"Configuration file not found at: {config_path.resolve()}")

    try:
        with open(config_path, 'r') as f:
            full_cfg = yaml.safe_load(f)

        cfg = full_cfg['MIMIC']

        root_dir = Path(cfg['root_dir']).resolve()
        data_dir = root_dir / cfg['data_subdir']
        res_dir = root_dir / cfg['output_subdir']

        root_files = cfg['root_files']
        patients_csv = root_dir / root_files['patients']
        admissions_csv = root_dir / root_files['admissions']
        chexpert_csv = root_dir / root_files['chexpert']
        metadata_csv = root_dir / root_files['metadata']

        return {
            'root_dir': root_dir,
            'data_dir': data_dir,
            'out_dir': res_dir,
            'patients_csv': patients_csv,
            'admissions_csv': admissions_csv,
            'chexpert_csv': chexpert_csv,
            'metadata_csv': metadata_csv
        }
    except Exception as e:
        raise Exception(f"Error loading preprocessing configuration {e} for config {config_path.resolve()}")

In [20]:
def load_and_merge_mimic_data():
    # Load preprocessing config
    config = load_config()

    patients = pd.read_csv(config['patients_csv'])
    patients = patients[['subject_id', 'gender']]

    admissions = pd.read_csv(config['admissions_csv'])
    admissions = admissions[['subject_id', 'insurance', 'ethnicity', 'marital_status']]
    admissions = admissions.drop_duplicates(subset=['subject_id'], keep='last')
    admissions = admissions.sort_values(by=['subject_id'])

    labels = pd.read_csv(config['chexpert_csv'])

    df = pd.read_csv(config['metadata_csv'])
    df = df[['dicom_id', 'subject_id', 'study_id', 'ViewPosition']]
    df.drop(df[(df['ViewPosition'] != 'PA') & (df['ViewPosition'] != 'AP')].index, inplace=True)

    df = pd.merge(df, patients)
    df = pd.merge(df, admissions)
    df = pd.merge(df, labels)

    df['path'] = 'files/p' + df['subject_id'].astype(str).str[:2] + '/p' + df['subject_id'].astype(str) + '/s' + df['study_id'].astype(str) + '/' + df['dicom_id'].astype(str) + '.jpg'

    return df

In [18]:
def process_image(img, transResize):
    # Center crop the image to square dimensions
    width, height = img.size
    r_min = max(0, (height - width) / 2)
    r_max = min(height, (height + width) / 2)
    c_min = max(0, (width - height) / 2)
    c_max = min(width, (width + height) / 2)
    img = img.crop((c_min, r_min, c_max, r_max))

    # Resize the image to the target size (transResize x transResize)
    img = img.resize((transResize, transResize))

    # Equalize histogram for contrast enhancement and convert to grayscale
    img = ImageOps.equalize(img)
    img = img.convert('L')

    return np.array(img)

# Loads and preprocesses a single image given its index and path.
# Returns a tuple of (index, processed_image_array), or None if any step fails.
def worker_process(index, filename, transResize, data_dir):
    try:
        img_path = os.path.join(data_dir, filename)
        img = Image.open(img_path).convert('RGB')
        img_array = process_image(img, transResize)
        return index, img_array
    except Exception as e:
        return None


def processAndSave(df, transResize=128, pool_size=28, chunk_size=1000):
    # Load preprocessing config
    config = load_config()
    
    # Prepare a memory-mapped .npy file to store resized image data efficiently;
    # creates the file if it doesn't exist, otherwise opens it for read/write access
    npy_output_path = os.path.join(config['out_dir'], f'files_{transResize}.npy')
    os.makedirs(os.path.dirname(npy_output_path), exist_ok=True)
    if not os.path.exists(npy_output_path):
        img_mat = np.memmap(npy_output_path, dtype='uint8', mode='w+', shape=(len(df), transResize, transResize))
    else:
        img_mat = np.memmap(npy_output_path, dtype='uint8', mode='r+', shape=(len(df), transResize, transResize))

    valid_indices = []
    processed_count = 0

    # Prepare arguments for multiprocessing
    args = [(i, fname, transResize, config['data_dir']) for i, fname in enumerate(df['path'])]

    with Pool(pool_size) as pool:
        pbar = tqdm(range(0, len(args), chunk_size), desc="Processing", dynamic_ncols=True)
        for i in pbar:
            chunk_start = time.perf_counter()

            batch = args[i:i + chunk_size]
            results = pool.starmap(worker_process, batch)

            for res in results:
                if res is None:
                    continue
                idx, img_array = res
                img_mat[len(valid_indices)] = img_array
                valid_indices.append(idx)
                processed_count += 1

            chunk_elapsed = time.perf_counter() - chunk_start
            item_time = chunk_elapsed / len(batch) if len(batch) > 0 else 0

            pbar.set_postfix({
                "Batch Time": f"{chunk_elapsed:.2f}s",
                "Item Time": f"{item_time:.4f}s",
                "Total": processed_count
            })

    # Finalize: flush memmap and save filtered metadata
    img_mat.flush()
    df2 = df.iloc[valid_indices].reset_index(drop=True)
    df2.to_csv(os.path.join(config['out_dir'], 'meta_data.csv'), index=False)




In [21]:
start = time.time()
df = load_and_merge_mimic_data()
processAndSave(df)
print(f"✅ Completed in {round((time.time() - start) / 60, 2)} minutes")

Processing:   0%|          | 0/218 [00:00<?, ?it/s]

✅ Completed in 27.68 minutes
